# Supplementary Figure S5 — patient-paired single-cell localization

Raw UMI counts define detection. Patient-compartment mean normalized log
expression defines the paired expression summaries. The inferential unit is
the patient, and the paired Wilcoxon tests are corrected across the two
prespecified genes displayed here.


In [1]:
from pathlib import Path
import hashlib
import os
import re
import tarfile
import warnings

import h5py
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, spearmanr, wilcoxon
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_5_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MITOXY_GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
MYELOID_GENES = ["MYD88", "TNF", "TLR2", "TLR4"]
SINGLE_CELL_FAMILY = [
    "BAX", "BAK1", "BID", "SLC7A11", "MYD88", "TNF",
    "TLR2", "TLR4", "RICTOR", "MTOR", "MAPKAP1",
]
ANGIO_GENES = ["VEGFA", "KDR", "ESM1", "PECAM1", "ANGPTL4", "CD34"]
TUMOR_MARKERS = ["CA9", "NDUFA4L2", "EGLN3", "KRT8", "KRT18", "EPCAM"]
MYELOID_MARKERS = ["LST1", "TYROBP", "FCER1G", "CTSS", "AIF1", "CD68"]

manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "single_cell_tar": source_path("GSE159115", "GSE159115_RAW.tar"),
    "single_cell_annotation": source_path(
        "GSE159115", "GSE159115_ccRCC_anno.csv.gz"
    ),
    "spatial_tar": source_path("GSE210041", "GSE210041_RAW.tar"),
    "javelin_workbook": source_path(
        "JAVELIN Renal 101", "clinical and molecular tables"
    ),
    "cptac_rna": source_path("CPTAC ccRCC", "tumor RNA expression"),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,single_cell_tar,/scratch/ar58064/mitoxy/data/raw/gse159115_pub...
1,single_cell_annotation,/scratch/ar58064/mitoxy/data/raw/gse159115_pub...
2,spatial_tar,/scratch/ar58064/mitoxy/data/raw/gse210041_pub...
3,javelin_workbook,/scratch/ar58064/mitoxy/data/raw/javelin_renal...
4,cptac_rna,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...


Public-source integrity audit: PASS


In [2]:
def decode_array(values):
    values = np.asarray(values)
    if values.dtype.kind in "SO":
        return np.array([
            value.decode() if isinstance(value, bytes) else str(value)
            for value in values
        ])
    return values.astype(str)


def zscore_frame(frame):
    frame = frame.astype(float)
    standard_deviation = frame.std(axis=0, ddof=0).replace(0, np.nan)
    result = frame.subtract(frame.mean(axis=0), axis=1).divide(
        standard_deviation, axis=1
    )
    if result.isna().any().any():
        bad = result.columns[result.isna().any()].tolist()
        raise ValueError(f"Cannot standardize genes with invalid variance: {bad}")
    return result


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    adjusted = np.full(values.shape, np.nan)
    adjusted[valid] = multipletests(values[valid], method="fdr_bh")[1]
    return adjusted


def fisher_interval(rho, n, alpha_z=1.96):
    z = np.arctanh(np.clip(float(rho), -0.999999, 0.999999))
    standard_error = 1 / np.sqrt(int(n) - 3)
    return (
        float(np.tanh(z - alpha_z * standard_error)),
        float(np.tanh(z + alpha_z * standard_error)),
    )


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def find_column(columns, include_any, required=True):
    terms = [normalized(term) for term in include_any]
    matches = [
        column for column in columns
        if any(term in normalized(column) for term in terms)
    ]
    if not matches:
        if required:
            raise KeyError(
                f"No column matched {include_any}. Available: {list(columns)}"
            )
        return None
    return sorted(matches, key=lambda value: (len(str(value)), str(value)))[0]


def read_10x_h5(path):
    with h5py.File(path, "r") as handle:
        if "matrix" in handle:
            group = handle["matrix"]
        elif all(key in handle for key in ["data", "indices", "indptr", "shape"]):
            group = handle
        else:
            compatible = [
                handle[key] for key in handle.keys()
                if isinstance(handle[key], h5py.Group)
                and all(
                    field in handle[key]
                    for field in ["data", "indices", "indptr", "shape"]
                )
            ]
            if len(compatible) != 1:
                raise RuntimeError(
                    f"Could not uniquely identify the 10x matrix group in {path}"
                )
            group = compatible[0]
        data = group["data"][:]
        indices = group["indices"][:]
        indptr = group["indptr"][:]
        shape = tuple(group["shape"][:])
        matrix = sparse.csc_matrix((data, indices, indptr), shape=shape)
        barcodes = decode_array(group["barcodes"][:])
        if "features" in group:
            features = group["features"]
            name_key = "name" if "name" in features else "gene_names"
            genes = decode_array(features[name_key][:]).astype(str)
        else:
            name_key = "gene_names" if "gene_names" in group else "genes"
            genes = decode_array(group[name_key][:]).astype(str)
    return matrix, genes, barcodes


def collapse_duplicate_genes(matrix, genes):
    genes = pd.Index(pd.Series(genes).astype(str).str.upper())
    unique = genes.unique()
    if len(unique) == len(genes):
        return matrix, unique.to_numpy()
    mapping = pd.Series(np.arange(len(unique)), index=unique)
    rows = mapping.loc[genes].to_numpy()
    aggregator = sparse.csr_matrix(
        (np.ones(len(rows)), (rows, np.arange(len(rows)))),
        shape=(len(unique), len(rows)),
    )
    return aggregator @ matrix, unique.to_numpy()


def selected_log_normalized(matrix, genes, selected_genes):
    matrix, genes = collapse_duplicate_genes(matrix, genes)
    gene_index = pd.Series(np.arange(len(genes)), index=genes)
    present = [gene for gene in selected_genes if gene in gene_index.index]
    missing = sorted(set(selected_genes) - set(present))
    if missing:
        warnings.warn(f"Genes absent from matrix and omitted: {missing}")
    library_size = np.asarray(matrix.sum(axis=0)).ravel().astype(float)
    library_size[library_size == 0] = np.nan
    rows = matrix[gene_index.loc[present].to_numpy(), :].toarray().T
    normalized_values = np.log1p(rows / library_size[:, None] * 10_000)
    return pd.DataFrame(normalized_values, columns=present), rows



## Panel A — GSE159115 paired single-cell compartment localization

Detection percentages use raw UMI counts. Patient-level tests use mean
library-size-normalized, log-transformed expression within each compartment,
followed by paired Wilcoxon tests and Benjamini–Hochberg correction across the
prespecified gene family.


In [4]:
single_cell_extract = RAW["single_cell_tar"].parent / "extracted"
h5_files = sorted(single_cell_extract.rglob("*.h5"))
if not h5_files:
    raise FileNotFoundError(
        f"No extracted GSE159115 H5 files under {single_cell_extract}. "
        "Rerun the GEO extraction cell in 00_Paper_Data_Download.ipynb."
    )

annotation = pd.read_csv(RAW["single_cell_annotation"], compression="gzip")
annotation.columns = [str(column).strip() for column in annotation.columns]

# The GEO annotation contains a barcode-bearing cell identifier. Resolve it
# from the public contents rather than assuming a release-specific name.
cell_id_scores = {}
for column in annotation.columns:
    values = annotation[column].astype(str)
    cell_id_scores[column] = values.str.contains(
        r"[ACGT]{14,18}", regex=True, na=False
    ).mean()
cell_id_column = max(cell_id_scores, key=cell_id_scores.get)
if cell_id_scores[cell_id_column] < 0.5:
    raise RuntimeError("Could not identify the annotation cell-barcode column")

# Select the lineage annotation, not a tissue/study column that happens to
# contain 'ccRCC' for every cell. A valid lineage column must contain both
# tumor-like and myeloid-like labels and should have a cell-type-like name.
myeloid_pattern = (
    r"myeloid|monocyte|macrophage|dendritic|(?:^|[^a-z])mono(?:[^a-z]|$)|"
    r"(?:^|[^a-z])macro(?:[^a-z]|$)|(?:^|[^a-z])dc(?:[^a-z]|$)"
)
tumor_pattern = r"tumou?r|malignant|cancer|ccrcc|epithelial|neoplastic"
lineage_pattern = (
    myeloid_pattern + r"|t[ -]?cell|b[ -]?cell|nk|endothelial|"
    r"fibroblast|pericyte|plasma|mast|erythro"
)
type_audit_rows = []
for column in annotation.columns:
    if column == cell_id_column:
        continue
    values = annotation[column].fillna("").astype(str).str.lower().str.strip()
    n_unique = int(values[values.ne("")].nunique())
    myeloid_hits = int(values.str.contains(myeloid_pattern, regex=True, na=False).sum())
    tumor_hits = int(values.str.contains(tumor_pattern, regex=True, na=False).sum())
    lineage_hits = int(values.str.contains(lineage_pattern, regex=True, na=False).sum())
    column_name = normalized(column)
    name_bonus = 0
    if any(term in column_name for term in [
        "cell type", "celltype", "annotation", "lineage", "identity", "ident"
    ]):
        name_bonus = 100
    elif "cluster" in column_name:
        name_bonus = 25
    usable_cardinality = 2 <= n_unique <= 200
    score = (
        name_bonus
        + 20 * int(myeloid_hits > 0)
        + 20 * int(tumor_hits > 0)
        + min(lineage_hits / max(len(values), 1), 1.0)
        + 5 * int(usable_cardinality)
    )
    type_audit_rows.append({
        "column": column, "n_unique": n_unique,
        "myeloid_hits": myeloid_hits, "tumor_hits": tumor_hits,
        "lineage_hits": lineage_hits, "score": score,
    })
type_audit = pd.DataFrame(type_audit_rows).sort_values("score", ascending=False)
valid_type_columns = type_audit.loc[
    type_audit["myeloid_hits"].gt(0)
    & type_audit["tumor_hits"].gt(0)
    & type_audit["n_unique"].between(2, 200)
]
if valid_type_columns.empty:
    display(type_audit.head(12))
    raise RuntimeError(
        "Could not identify a lineage column containing both tumor and "
        "myeloid annotations"
    )
cell_type_column = valid_type_columns.iloc[0]["column"]
print("Resolved cell-type column:", cell_type_column)
display(type_audit.head(8))

annotation_keys = annotation[cell_id_column].astype(str)
annotation_barcode = annotation_keys.str.extract(
    r"([ACGT]{14,18}(?:-\d+)?)", expand=False
).str.replace(r"-\d+$", "", regex=True)

cell_blocks = []
for h5_path in h5_files:
    matrix, genes, barcodes = read_10x_h5(h5_path)
    values, raw_selected = selected_log_normalized(
        matrix, genes, SINGLE_CELL_FAMILY
    )
    barcode_base = pd.Series(barcodes).str.replace(
        r"-\d+$", "", regex=True
    )
    sample_tokens = re.findall(r"SI[_-]?\d+|GSM\d+", h5_path.name, flags=re.I)
    sample_token = sample_tokens[0] if sample_tokens else h5_path.stem

    candidate = annotation.loc[annotation_barcode.isin(set(barcode_base))].copy()
    if sample_tokens:
        token_mask = candidate.astype(str).apply(
            lambda column: column.str.contains(
                sample_token, case=False, regex=False, na=False
            )
        ).any(axis=1)
        if token_mask.any():
            candidate = candidate.loc[token_mask].copy()

    candidate["_barcode_base"] = annotation_barcode.loc[candidate.index]
    candidate = candidate.drop_duplicates("_barcode_base")
    metadata = pd.DataFrame({
        "_barcode_base": barcode_base,
        "_row": np.arange(len(barcodes)),
    }).merge(candidate, on="_barcode_base", how="inner")
    if metadata.empty:
        warnings.warn(f"No annotation overlap for {h5_path.name}; skipping")
        continue

    block = values.iloc[metadata["_row"].to_numpy()].reset_index(drop=True)
    raw_block = raw_selected[metadata["_row"].to_numpy(), :]
    for index, gene in enumerate(values.columns):
        block[f"detected_{gene}"] = raw_block[:, index] > 0
    block["patient"] = sample_token
    block["cell_type"] = metadata[cell_type_column].astype(str).to_numpy()
    cell_blocks.append(block)

if not cell_blocks:
    raise RuntimeError("No annotated GSE159115 ccRCC cells were reconstructed")
single_cells = pd.concat(cell_blocks, ignore_index=True)
type_text = single_cells["cell_type"].fillna("").astype(str).str.lower().str.strip()
single_cells["compartment"] = np.select(
    [
        type_text.str.contains(myeloid_pattern, regex=True, na=False),
        type_text.str.contains(tumor_pattern, regex=True, na=False),
    ],
    ["Myeloid", "Tumor"],
    default="Other",
)
compartment_counts = single_cells["compartment"].value_counts(dropna=False)
print("Reconstructed compartment counts:")
display(compartment_counts.rename("n_cells").to_frame())
missing_compartments = sorted(
    {"Tumor", "Myeloid"} - set(compartment_counts.index)
)
if missing_compartments:
    display(single_cells["cell_type"].value_counts().head(30))
    raise RuntimeError(
        f"Missing reconstructed compartments: {missing_compartments}; "
        f"cell-type column was {cell_type_column!r}"
    )
analysis_cells = single_cells.loc[
    single_cells["compartment"].isin(["Tumor", "Myeloid"])
].copy()

paired_rows = []
for gene in SINGLE_CELL_FAMILY:
    if gene not in analysis_cells.columns:
        continue
    patient_compartment = (
        analysis_cells.pivot_table(
            index="patient", columns="compartment",
            values=gene, aggfunc="mean"
        )
        .reindex(columns=["Tumor", "Myeloid"])
        .dropna()
    )
    if len(patient_compartment) < 3:
        raise RuntimeError(
            f"Too few paired patients for {gene}: {len(patient_compartment)}"
        )
    statistic, p_value = wilcoxon(
        patient_compartment["Tumor"],
        patient_compartment["Myeloid"],
        alternative="two-sided",
        zero_method="wilcox",
    )
    paired_rows.append({
        "gene": gene,
        "n_patients": len(patient_compartment),
        "paired_p": float(p_value),
    })

paired = pd.DataFrame(paired_rows)
paired["paired_fdr"] = bh_adjust(paired["paired_p"])

localization_rows = []
for gene in ["MYD88", "TNF", "TLR2", "TLR4", "RICTOR"]:
    localization_rows.append({
        "gene": gene,
        "tumor_detection_pct": 100 * analysis_cells.loc[
            analysis_cells["compartment"].eq("Tumor"), f"detected_{gene}"
        ].mean(),
        "myeloid_detection_pct": 100 * analysis_cells.loc[
            analysis_cells["compartment"].eq("Myeloid"), f"detected_{gene}"
        ].mean(),
    })

panel_a = pd.DataFrame(localization_rows).merge(
    paired[["gene", "paired_fdr"]], on="gene", how="left",
    validate="one_to_one",
)
if panel_a.isna().any().any():
    raise ValueError("Panel A contains missing reconstructed values")
panel_a.to_csv(OUTPUT_DIR / "panel_a_compartment_localization.csv", index=False)

print("Annotated ccRCC cells:", len(single_cells))
print("Tumor/myeloid cells retained:", len(analysis_cells))
print("Paired patients:", paired["n_patients"].min())
display(panel_a)



Resolved cell-type column: label


,column,n_unique,myeloid_hits,tumor_hits,lineage_hits,score
11,label,14,4186,9139,5602,45.270002
10,anno,13,4186,9139,5602,45.270002
9,cluster,14,0,0,0,30.000000
12,patient,7,0,0,0,5.000000
3,sample,7,0,0,0,5.000000
4,gene_per_1kUMI,20629,0,0,0,0.000000
2,pct_MT,20456,0,0,0,0.000000
1,no_genes,4604,0,0,0,0.000000


Reconstructed compartment counts:


,n_cells
compartment,
Tumor,9285
Other,7510
Myeloid,4295


Annotated ccRCC cells: 21090
Tumor/myeloid cells retained: 13580
Paired patients: 14


,gene,tumor_detection_pct,myeloid_detection_pct,paired_fdr
0,MYD88,8.002154,19.557625,0.148779
1,TNF,2.627894,26.100116,0.194048
2,TLR2,3.478729,39.115250,0.148779
3,TLR4,1.249327,15.599534,0.148779
4,RICTOR,25.277329,7.916182,0.148779


In [5]:
OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/"supplementary_figure_s5"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
genes=["BAX","BID"]; paired_long=[]; summary=[]
for gene in genes:
    expression=analysis_cells.pivot_table(index="patient",columns="compartment",values=gene,aggfunc="mean").reindex(columns=["Tumor","Myeloid"]).dropna()
    detection=analysis_cells.pivot_table(index="patient",columns="compartment",values=f"detected_{gene}",aggfunc="mean").reindex(columns=["Tumor","Myeloid"]).dropna()
    common=expression.index.intersection(detection.index); expression=expression.loc[common]; detection=detection.loc[common]
    stat,p=wilcoxon(expression.Tumor,expression.Myeloid,alternative="two-sided",zero_method="wilcox")
    for patient_id in common:
        for compartment in ["Tumor","Myeloid"]:
            paired_long.append({"patient":patient_id,"gene":gene,"compartment":compartment,
                                "mean_log_expression":expression.loc[patient_id,compartment],
                                "detection_fraction":detection.loc[patient_id,compartment]})
    summary.append({"gene":gene,"n_patients":len(common),"tumor_expression_mean":expression.Tumor.mean(),
                    "myeloid_expression_mean":expression.Myeloid.mean(),"median_paired_delta":(expression.Tumor-expression.Myeloid).median(),
                    "tumor_detection_pct":100*detection.Tumor.mean(),"myeloid_detection_pct":100*detection.Myeloid.mean(),"paired_p":p})
panel_a_data=pd.DataFrame(paired_long); panel_a_summary=pd.DataFrame(summary)
panel_a_summary["paired_fdr"]=multipletests(panel_a_summary.paired_p,method="fdr_bh")[1]
panel_a_data.to_csv(OUTPUT_DIR/"panel_a_paired_data.csv",index=False); panel_a_summary.to_csv(OUTPUT_DIR/"panel_a_summary.csv",index=False)
display(panel_a_summary); display(panel_a_data.head())



,gene,n_patients,tumor_expression_mean,myeloid_expression_mean,median_paired_delta,tumor_detection_pct,myeloid_detection_pct,paired_p,paired_fdr
0,BAX,14,0.355054,0.404224,-0.082694,36.013061,39.362964,0.357544,0.357544
1,BID,14,0.111206,0.272174,-0.219185,13.134687,26.276863,0.067627,0.135254


,patient,gene,compartment,mean_log_expression,detection_fraction
0,GSM4819725,BAX,Tumor,0.441163,0.551759
1,GSM4819725,BAX,Myeloid,0.519552,0.626437
2,GSM4819726,BAX,Tumor,0.086682,0.105263
3,GSM4819726,BAX,Myeloid,0.192583,0.333333
4,GSM4819727,BAX,Tumor,0.337178,0.365979


In [6]:
contracts = {'panel_a_paired_data.csv': ['patient', 'gene', 'compartment', 'mean_log_expression', 'detection_fraction'], 'panel_a_summary.csv': ['gene', 'n_patients', 'tumor_expression_mean', 'myeloid_expression_mean', 'tumor_detection_pct', 'myeloid_detection_pct', 'paired_p', 'paired_fdr']}
rows=[]
for filename,required in contracts.items():
    path=OUTPUT_DIR/filename
    table=pd.read_csv(path)
    missing=sorted(set(required)-set(table.columns))
    if missing: raise KeyError(f"{filename} missing {missing}")
    rows.append({"file":filename,"rows":len(table),"size_bytes":path.stat().st_size,"sha256":sha256_file(path)})
processed_manifest=pd.DataFrame(rows)
processed_manifest.to_csv(OUTPUT_DIR/"processed_manifest.csv",index=False)
display(processed_manifest)
print("Processed-file handoff: PASS")



,file,rows,size_bytes,sha256
0,panel_a_paired_data.csv,56,3158,fec1956e70bb180ea59d747eef6a541198b4a3a50c0679...
1,panel_a_summary.csv,2,411,c25c47b8ca12e2c67db8d92ebafdc2b36bfeaad79d2103...


Processed-file handoff: PASS
